# M3 profile: flight-delay-weather-analysis

Profiles the M2 database (`data/flights.db`) before any cleaning decision. Read-only: this notebook never writes to the database.
Run it top to bottom on a fresh build (`python src/load.py`). All SQL here is fixed text with no formatted values.

Tables: `aircraft`, `aircraft_state` (OpenSky), `location`, `weather_hourly` (Open-Meteo), `quarantine`.
Units as delivered: OpenSky altitude in **m**, velocity in **m/s**, vertical rate in **m/s**; Open-Meteo wind in **km/h**, precipitation in **mm**, CAPE in **J/kg**.

In [ ]:
import sqlite3, sys, unicodedata
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
from db import connect, DB_PATH

pd.set_option("display.width", 160, "display.max_columns", 40)
con = connect(DB_PATH)   # foreign_keys=ON; we only SELECT

SQL_STATE = "SELECT s.*, a.icao24, a.origin_country FROM aircraft_state s JOIN aircraft a USING (aircraft_key)"
SQL_WEATHER = "SELECT w.*, l.location_index, l.latitude, l.longitude FROM weather_hourly w JOIN location l USING (location_key)"
state = pd.read_sql_query(SQL_STATE, con)
weather = pd.read_sql_query(SQL_WEATHER, con)
aircraft = pd.read_sql_query("SELECT * FROM aircraft", con)
location = pd.read_sql_query("SELECT * FROM location", con)
print("database:", DB_PATH)
print({"aircraft": len(aircraft), "aircraft_state": len(state), "location": len(location),
       "weather_hourly": len(weather),
       "quarantine": con.execute("SELECT COUNT(*) FROM quarantine").fetchone()[0]})

## 1. Row counts, and rows per entity
Unequal coverage makes comparisons meaningless.

In [ ]:
print("weather rows per location:")
print(weather.groupby("location_index").size().rename("hours").to_frame().T)
per_ac = state.groupby("aircraft_key").size()
print("\nstates per aircraft (distribution):")
print(per_ac.value_counts().sort_index().rename("aircraft").to_frame().T)
print("\nstates per snapshot (retrieved_at):")
print(state.groupby("retrieved_at").size())
print("\nstates per origin_country (top 10):")
print(state.groupby("origin_country").size().sort_values(ascending=False).head(10))

## 2. Missing values: count and share, per entity and over time
NULL means 'not reported'. Look for missingness that depends on something else (on the ground? which snapshot?).

In [ ]:
cols = ["callsign","longitude","latitude","baro_altitude","geo_altitude","velocity","true_track",
        "vertical_rate","squawk","category","last_contact"]
miss = pd.DataFrame({"missing": state[cols].isna().sum(), "share": state[cols].isna().mean().round(3)})
print(miss)
print("\nMissing share by on_ground (0 = airborne, 1 = on ground):")
print(state.groupby("on_ground")[["baro_altitude","geo_altitude","vertical_rate","squawk"]]
      .apply(lambda g: g.isna().mean().round(2)))
print("\nMissing share by snapshot:")
print(state.groupby("retrieved_at")[["baro_altitude","geo_altitude","vertical_rate","squawk"]]
      .apply(lambda g: g.isna().mean().round(2)))
print("\nEmpty-string callsigns (not NULL, but nothing in them):",
      int((state.callsign.fillna("").str.strip() == "").sum()))
print("\nWeather NULLs per variable:")
print(weather[["wind_speed_10m","wind_gusts_10m","precipitation","showers","cape"]].isna().sum())

## 3. Extremes and most frequent values
Sentinels (-999, 0, 9999), negatives and impossible values hide in the extremes and the modes.

In [ ]:
num_state = ["longitude","latitude","baro_altitude","geo_altitude","velocity","true_track","vertical_rate"]
print(state[num_state].agg(["min","median","max"]).T)
print("\nMost frequent baro_altitude values on the ground (a repeated odd value is a sentinel candidate):")
print(state[state.on_ground == 1].baro_altitude.value_counts().head(6))
print(state.loc[state.baro_altitude < 0, ["icao24","time_position","on_ground","baro_altitude","geo_altitude","velocity","latitude","longitude"]])
print("\nMost frequent values:")
for c in ["baro_altitude","velocity","vertical_rate","squawk"]:
    print(c, state[c].value_counts().head(4).to_dict())
print("\nSentinel candidates (-999, -9999, 9999, 99999) in OpenSky numeric columns:",
      int(state[num_state].isin([-999, -9999, 9999, 99999]).sum().sum()))

num_w = ["wind_speed_10m","wind_gusts_10m","precipitation","showers","cape"]
print("\n", weather.groupby("location_index")[num_w].agg(["min","median","max"]).T)
print("\nSentinel candidates in weather:", int(weather[num_w].isin([-999, -9999, 9999, 99999]).sum().sum()))

## 4. Text columns: distinct values with repr()
Two strings that print the same may not be the same.

In [ ]:
print("callsign samples:", [repr(v) for v in state.callsign.dropna().unique()[:6]])
print("callsigns with trailing/leading whitespace:",
      int((state.callsign.dropna() != state.callsign.dropna().str.strip()).sum()))
print("callsigns with lowercase letters:", int(state.callsign.dropna().str.contains("[a-z]").sum()))
print("distinct squawk lengths:", state.squawk.dropna().str.len().value_counts().to_dict())
countries = aircraft.origin_country.dropna().unique()
print("\norigin_country: %d distinct" % len(countries))
print("not NFC-normalised:", [repr(x) for x in countries if unicodedata.normalize("NFC", x) != x])
print("with non-ASCII characters:", [repr(x) for x in countries if not x.isascii()])
print("same after casefold/strip but different strings:",
      pd.Series(countries).groupby(pd.Series(countries).str.strip().str.casefold()).filter(lambda g: len(g) > 1).tolist())
print("\nicao24 format ok (6 hex chars):", bool(aircraft.icao24.str.fullmatch("[0-9a-f]{6}").all()))

## 5. First and last time each entity appears
An entity that stops when another starts may be the same entity.

In [ ]:
print(weather.groupby("location_index").observed_at.agg(["min","max","count"]))
print("\nWeather days present:", sorted(weather.observed_at.str[:10].unique()))
span = state.groupby("aircraft_key").time_position.agg(["min","max","count"])
print("\nAircraft seen in more than one snapshot:", int((state.groupby("aircraft_key").retrieved_at.nunique() > 1).sum()),
      "of", len(span))
print("\nSnapshots (retrieved_at) and the span of their time_position:")
print(state.groupby("retrieved_at").time_position.agg(["min","max","count"]))

## 6. Values in sequence: stuck values and spikes
Stuck values, spikes and level shifts pass every summary statistic.

In [ ]:
def longest_run(series):
    best = cur = 1
    for a, b in zip(series.values[:-1], series.values[1:]):
        cur = cur + 1 if a == b else 1
        best = max(best, cur)
    return best

w = weather.sort_values(["location_index", "observed_at"])
runs = (w.groupby("location_index")[["wind_speed_10m","wind_gusts_10m","cape"]]
          .agg(longest_run).rename(columns=lambda c: "longest_identical_run_" + c))
print(runs)
print("(note: 0.0 is a natural repeated value for cape and precipitation; wind runs are the informative ones)")

w["d_speed"] = w.groupby("location_index").wind_speed_10m.diff().abs()
print("\nLargest hour-to-hour wind speed change per location (km/h):")
print(w.groupby("location_index").d_speed.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for loc, g in w.groupby("location_index"):
    axes[0].plot(range(len(g)), g.wind_gusts_10m.values, label=str(loc), lw=1)
    axes[1].plot(range(len(g)), g.cape.values, label=str(loc), lw=1)
axes[0].set_title("wind_gusts_10m by location (hours in order)"); axes[1].set_title("cape by location")
axes[0].legend(title="location_index"); plt.tight_layout(); plt.show()

## 7. Invalid values (family 2): rules
Each rule is a query that must return 0 if the data is valid. Rules are about what cannot be true.

In [ ]:
BBOX = (36.0, 26.0, 42.0, 45.0)   # lamin, lomin, lamax, lomax from OPENSKY_BBOX
checks = {
    "latitude outside bbox": int(((state.latitude < BBOX[0]) | (state.latitude > BBOX[2])).sum()),
    "longitude outside bbox": int(((state.longitude < BBOX[1]) | (state.longitude > BBOX[3])).sum()),
    "negative velocity": int((state.velocity < 0).sum()),
    "velocity above 400 m/s": int((state.velocity > 400).sum()),
    "vertical rate beyond +-100 m/s": int((state.vertical_rate.abs() > 100).sum()),
    "true_track outside 0..360": int(((state.true_track < 0) | (state.true_track > 360)).sum()),
    "negative baro_altitude (see section 3: -304.8 m = -1000 ft)": int((state.baro_altitude < 0).sum()),
    "baro_altitude above 20,000 m": int((state.baro_altitude > 20000).sum()),
    "last_contact before time_position": int((state.last_contact < state.time_position).sum()),
    "time_position later than retrieved_at (any amount)": int((state.time_position > state.retrieved_at).sum()),
    "time_position more than 60 s later than retrieved_at": int(((pd.to_datetime(state.time_position) - pd.to_datetime(state.retrieved_at)).dt.total_seconds() > 60).sum()),
    "squawk not 4 octal digits": int((~state.squawk.dropna().str.fullmatch("[0-7]{4}")).sum()),
    "weather: negative values": int((weather[["wind_speed_10m","wind_gusts_10m","precipitation","showers","cape"]] < 0).sum().sum()),
    "weather: gust below mean wind speed": int((weather.wind_gusts_10m < weather.wind_speed_10m).sum()),
    "weather: showers above precipitation": int((weather.showers > weather.precipitation).sum()),
}
print(pd.Series(checks, name="rows violating the rule").to_string())
print("\nStates with an impossible velocity or vertical rate:")
print(state.loc[(state.velocity > 400) | (state.vertical_rate.abs() > 100),
                ["icao24","callsign","time_position","on_ground","velocity","vertical_rate","baro_altitude","geo_altitude"]])

## 8. Failures across fields (family 4): compare columns

In [ ]:
air = state[state.on_ground == 0]
gnd = state[state.on_ground == 1]
print("On ground:", len(gnd), "rows; of these baro_altitude NULL:", int(gnd.baro_altitude.isna().sum()),
      "; velocity above 50 m/s (180 km/h):", int((gnd.velocity > 50).sum()))
print("Airborne rows with NULL or <= 0 baro_altitude:", int((air.baro_altitude.isna() | (air.baro_altitude <= 0)).sum()))
state["alt_gap"] = (state.geo_altitude - state.baro_altitude).abs()
print("\n|geo_altitude - baro_altitude| (m): median %.0f, 95th pct %.0f, max %.0f" % (
      state.alt_gap.median(), state.alt_gap.quantile(.95), state.alt_gap.max()))
print("Rows with a gap above 1,000 m:", int((state.alt_gap > 1000).sum()))
print(state.loc[state.alt_gap > 1000, ["icao24","on_ground","baro_altitude","geo_altitude","velocity","retrieved_at"]].head(8))
print("\nspi = 1 (special purpose indicator) rows:", int((state.spi == 1).sum()))
age = (pd.to_datetime(state.retrieved_at) - pd.to_datetime(state.time_position)).dt.total_seconds()
print("\nAge of a state at retrieval (s): median %.0f, 95th pct %.0f, max %.0f, min %.0f" % (
      age.median(), age.quantile(.95), age.max(), age.min()))

### 8b. How old are the states, and how do the two altitudes disagree?

In [ ]:
ts = pd.to_datetime(state.retrieved_at) - pd.to_datetime(state.time_position)
age_s = ts.dt.total_seconds()
print("Age of a state at retrieval (s): 50th %.0f, 95th %.0f, 99th %.0f, max %.0f" % (
      age_s.quantile(.5), age_s.quantile(.95), age_s.quantile(.99), age_s.max()))
print("states older than 5 min:", int((age_s > 300).sum()), "; older than 15 min:", int((age_s > 900).sum()))
print("time_position later than retrieved_at: %d states, by at most %.0f s" % (
      int((age_s < 0).sum()), -age_s.min()))
mm = state[state.alt_gap > 1000].assign(gap_m=lambda d: (d.geo_altitude - d.baro_altitude).round(0))
print("\nAltitude mismatches (> 1,000 m): %d of %d states with both altitudes" % (
      len(mm), int((state.baro_altitude.notna() & state.geo_altitude.notna()).sum())))
print(mm.groupby("retrieved_at").size().rename("per snapshot").to_frame().T)
print(mm[["icao24","callsign","baro_altitude","geo_altitude","gap_m","velocity","vertical_rate"]].sort_values("gap_m"))

## 9. Unusual values (family 3): robust methods, per entity
IQR fences and robust z-score (median and MAD), per location. Flag and look; do not delete. Never mean ± 3 SD.

In [ ]:
def iqr_flag(g, col):
    q1, q3 = g[col].quantile([.25, .75]); iqr = q3 - q1
    return (g[col] < q1 - 1.5 * iqr) | (g[col] > q3 + 1.5 * iqr)

def robust_z(g, col):
    med = g[col].median(); mad = (g[col] - med).abs().median()
    return (g[col] - med) / (1.4826 * mad) if mad > 0 else g[col] * 0

out = []
for col in ["wind_gusts_10m", "cape"]:
    for loc, g in weather.groupby("location_index"):
        out.append({"variable": col, "location_index": loc,
                    "iqr_flagged": int(iqr_flag(g, col).sum()),
                    "robust_z_above_3.5": int((robust_z(g, col) > 3.5).sum()),
                    "max": g[col].max()})
print(pd.DataFrame(out).to_string(index=False))
print("(cape is 0 most of the time, so the IQR is 0 and every non-zero hour is flagged, while MAD = 0 gives no robust z at all: "
      "these methods need a different treatment for zero-inflated variables)")
print("\nLook at what is flagged before believing it (hours of the highest gusts):")
print(weather.sort_values("wind_gusts_10m", ascending=False)[["location_index","observed_at","wind_speed_10m","wind_gusts_10m","cape"]].head(8))
print("\nAircraft velocity per origin: IQR flagged (all aircraft together):",
      int(iqr_flag(state.dropna(subset=['velocity']), 'velocity').sum()))

### 9b. Look at what the robust z-score flags
A flag is only believable if it makes sense: do the same days light up at several airports (a real weather event), and when in the day do they occur?

In [ ]:
wz = weather.copy()
wz["flag"] = False
for loc, g in wz.groupby("location_index"):
    wz.loc[g.index, "flag"] = robust_z(g, "wind_gusts_10m") > 3.5
fl = wz[wz.flag]
print("Flagged gust hours per day and location (location_index):")
print(fl.groupby([fl.observed_at.str[:10], "location_index"]).size().unstack(fill_value=0))
print("\nFlagged gust hours by hour of day (UTC):")
print(fl.groupby(fl.observed_at.str[11:13]).size().to_frame("flagged").T)
print("\nFlagged hours with gusts below 25 km/h, per location:")
print(fl[fl.wind_gusts_10m < 25].groupby("location_index").size())

## 10. Identity (family 6): one thing as two, two as one

In [ ]:
cs = state.assign(cs=state.callsign.str.strip()).dropna(subset=["cs"])
cs = cs[cs.cs != ""]
multi_cs = cs.groupby("icao24").cs.nunique()
print("icao24 with more than one callsign:", int((multi_cs > 1).sum()), "of", cs.icao24.nunique())
print(cs[cs.icao24.isin(multi_cs[multi_cs > 1].index[:3])].groupby("icao24").cs.unique())
multi_ac = cs.groupby("cs").icao24.nunique()
print("\ncallsigns used by more than one icao24:", int((multi_ac > 1).sum()))
print(multi_ac[multi_ac > 1].head(8))
print("\nlocations (should be 5 distinct places):")
print(location[["location_index","latitude","longitude","elevation"]])
print("\nicao24 that appear as both airborne and on-ground in different rows:",
      int((state.groupby("icao24").on_ground.nunique() > 1).sum()))

### 10b. Snapshot times and coverage

In [ ]:
print("Snapshots per day:")
print(state.groupby(state.retrieved_at.str[:10]).retrieved_at.nunique().to_frame("snapshots").T)
print("\nHour of day (UTC) of every snapshot:", sorted({t[11:13] for t in state.retrieved_at.unique()}))
print("\nDays with weather:", sorted(weather.observed_at.str[:10].unique()))
print("\nWeather rows whose hour is later than the file that supplied them (still forecasts):",
      int((weather.observed_at.str[:13] > weather.retrieved_at.str[:13]).sum()), "of", len(weather))

## 11. Notes to carry into the issues register
Fill in after running (this cell is for the team): which checks found something, which found nothing (write the check name and the result), and a first guess for the cause with confirmed / inferred / unknown.